In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from mlxtend import __version__ as mlxtend_version
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

from scipy.stats import pearsonr

DATASET_FOLDER = "dataset/"

In [ ]:
artists = pd.read_csv(DATASET_FOLDER + 'artists.csv', sep=';')
tracks = pd.read_csv(DATASET_FOLDER + 'tracks.csv', sep=',')

artists_p = artists.add_prefix("artist_")
tracks_p = tracks.add_prefix("track_")

merged = tracks_p.merge(
    artists_p,
    how="left",
    left_on="track_id_artist",
    right_on="artist_id_author")

pair_cols = ["track_id", "artist_id_author"]
merged = merged.drop_duplicates(subset=pair_cols, keep="first")

**Discretization strategy (justification)**
- BPM: cut at 90 and 120 BPM, common musicology thresholds for slow/medium/fast tempos.
- Popularity: use 0–30–70–100 to map to low/medium/high on a 0–100 scale.
- Duration: short < 3 min, medium 3–5 min, long > 5 min (typical radio lengths).
- Swear density: compute swear words per minute; split low/high by median to avoid skew from heavy tails.

In [ ]:
# Discretize numeric variables into categorical bins
bpm = pd.to_numeric(merged["track_bpm"], errors="coerce")
pop = pd.to_numeric(merged["track_popularity"], errors="coerce")
dur = pd.to_numeric(merged["track_duration_ms"], errors="coerce")
swear_it = pd.to_numeric(merged["track_swear_IT"], errors="coerce")
swear_en = pd.to_numeric(merged["track_swear_EN"], errors="coerce")
swear_total = swear_it.add(swear_en, fill_value=0)

# Swear density = swear words per minute
duration_min = dur / 60000
swear_density = swear_total / duration_min
swear_density = swear_density.replace([np.inf, -np.inf], np.nan)

encoded = pd.get_dummies([])

encoded["bpm"] = pd.cut(
    bpm,
    bins=[-np.inf, 90, 120, np.inf],
    labels=["Slow", "Medium", "Fast"]
 )

encoded["popularity"] = pd.cut(
    pop,
    bins=[-np.inf, 3e1, np.inf],
    labels=["Low", "High"]
 )

encoded["duration"] = pd.cut(
    dur,
    bins=[-np.inf, 180000, 300000, np.inf],
    labels=["Short", "Medium", "Long"]
 )

median_swear_density = swear_density.median(skipna=True)
encoded["swear_density"] = pd.cut(
    swear_density,
    bins=[-np.inf, median_swear_density, np.inf],
    labels=["Low", "High"]
 )

encoded["lexical_density"] = pd.cut(
    pd.to_numeric(merged["track_lexical_density"], errors="coerce"),
    bins=[-np.inf, 0.5, np.inf],
    labels=["Low", "High"]
 )

 
encoded["loudness"] = pd.cut(
    pd.to_numeric(merged["track_loudness"], errors="coerce"),
    bins=[-np.inf, 27, 54, np.inf],
    labels=["Soft", "Medium", "Loud"]
 )

encoded["acuteness"] = pd.cut(
    pd.to_numeric(merged["track_centroid"], errors="coerce"),
    bins=[-np.inf, 0.15, np.inf],
    labels=["Low", "High"]
 )

encoded.head()

C:\Users\ludol\AppData\Local\Temp\ipykernel_57784\306448593.py:14: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  encoded = pd.get_dummies([])


,bpm,popularity,duration,swear_density,lexical_density,loudness,acuteness
0,Fast,High,Medium,High,High,Soft,High
1,Fast,High,Medium,High,High,Soft,High
2,Fast,High,Medium,High,High,Medium,High
3,Fast,High,Short,High,High,Soft,Low
4,Medium,High,Medium,Low,Low,Soft,Low


In [ ]:
# Build transactions with explicit col_name_value labels
transactions = encoded.apply(lambda row: [f"{col}_{row[col]}" for col in encoded.columns], axis=1).tolist()
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

df.head()



,acuteness_High,acuteness_Low,acuteness_nan,bpm_Fast,bpm_Medium,bpm_Slow,bpm_nan,duration_Long,duration_Medium,duration_Short,...,loudness_Loud,loudness_Medium,loudness_Soft,loudness_nan,popularity_High,popularity_Low,popularity_nan,swear_density_High,swear_density_Low,swear_density_nan
0,True,False,False,True,False,False,False,False,True,False,...,False,False,True,False,True,False,False,True,False,False
1,True,False,False,True,False,False,False,False,True,False,...,False,False,True,False,True,False,False,True,False,False
2,True,False,False,True,False,False,False,False,True,False,...,False,True,False,False,True,False,False,True,False,False
3,False,True,False,True,False,False,False,False,False,True,...,False,False,True,False,True,False,False,True,False,False
4,False,True,False,False,True,False,False,False,True,False,...,False,False,True,False,True,False,False,False,True,False


In [ ]:
frequent_itemsets = apriori(df, min_support=2e-1, use_colnames=True)
frequent_itemsets.sort_values("support", ascending=False)

,support,itemsets
1,0.683593,(acuteness_Low)
5,0.620724,(duration_Medium)
10,0.610245,(loudness_Soft)
7,0.598872,(lexical_density_High)
11,0.521942,(popularity_High)
...,...,...
87,0.208490,"(duration_Medium, loudness_Soft, swear_density..."
94,0.208490,"(swear_low_soft, duration_Medium, loudness_Sof..."
4,0.207147,(bpm_Slow)
56,0.202669,"(popularity_Low, lexical_density_Low)"


In [ ]:
rules_conf = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.7,
    num_itemsets=len(df.index),
)

columns_to_show = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "leverage",
]

rules_conf[columns_to_show].sort_values(["lift", "leverage"], ascending=False).head(20)

,antecedents,consequents,support,confidence,lift,leverage
383,"(bpm_Fast, loudness_Medium, duration_Short, po...",(swear_density_High),0.022389,0.728863,1.468245,0.007140
158,"(loudness_Medium, duration_Short, lexical_dens...",(swear_density_High),0.025076,0.710660,1.431576,0.007560
161,"(duration_Short, popularity_High, lexical_dens...",(swear_density_High),0.029106,0.704989,1.420153,0.008611
163,"(loudness_Medium, popularity_High, lexical_den...",(swear_density_High),0.032330,0.700971,1.412059,0.009434
382,"(swear_density_High, bpm_Fast, loudness_Medium...",(popularity_High),0.022389,0.735294,1.408767,0.006496
378,"(swear_density_High, acuteness_Low, loudness_M...",(popularity_High),0.028211,0.727483,1.393801,0.007971
337,"(acuteness_Low, bpm_Fast, loudness_Medium, dur...",(popularity_High),0.020509,0.722397,1.384058,0.005691
302,"(bpm_Fast, loudness_Medium, swear_density_High...",(popularity_High),0.028211,0.714286,1.368517,0.007597
179,"(bpm_Fast, loudness_Medium, duration_Short, ac...",(popularity_High),0.028838,0.707692,1.355884,0.007569
297,"(bpm_Fast, loudness_Medium, duration_Short, le...",(popularity_High),0.030718,0.701431,1.343889,0.007861


In [ ]:
df["swear_high_short"] = df["swear_density_High"] & df["duration_Short"]
df["swear_low_soft"] = df["swear_density_Low"] & df["loudness_Soft"]
df["swear_low_popular"] = df["swear_density_Low"] & df["popularity_High"]
df["lexical_low_soft"] = df["lexical_density_Low"] & df["loudness_Soft"]